In [ ]:
!pip install -r requirements.txt -q

In [ ]:
import os
import io
import json
import requests
import base64
import psycopg2
from datetime import datetime
from minio import Minio
from pymilvus import connections, Collection, utility
from PIL import Image
import base64

# Configuration
HELM_FULLNAME = "insurguard"

# MinIO
MINIO_ENDPOINT   = f"{HELM_FULLNAME}-minio:9010"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
MINIO_BUCKET     = "kyc-data"

# PostgreSQL
POSTGRES_DSN = f"postgresql://postgres:postgres_password@{HELM_FULLNAME}-postgres:5432/kyc_db"

# Milvus
MILVUS_HOST       = f"{HELM_FULLNAME}-milvus"
MILVUS_PORT       = "19530"
MILVUS_COLLECTION = "kyc_face_embeddings"

# Application services
KYC_VERIFIER_URL   = f"http://{HELM_FULLNAME}-kyc-verifier:8000"
FACE_EMBEDDING_URL = f"http://{HELM_FULLNAME}-face-embedding-api:8000/embed"

# OCR service — NemoRetriever OCR v1 NIM
OCR_URL = f"http://{HELM_FULLNAME}-nemoretriever-parser:8000/v1/infer"

print("✅ Imports complete")

In [ ]:
import base64
import json
import requests
import logging
from minio import Minio
from dataclasses import dataclass, field
from typing import Optional
from datetime import date
from tenacity import retry, stop_after_attempt, wait_exponential

def get_image_bytes(minio_client: Minio, path: str) -> bytes:
    """
    Fetch image bytes from S3 URI, HTTP URL, or Data URI.
    
    Synchronous version converted from kyc-verifier/src/services/storage.py
    
    Args:
        minio_client: MinIO client instance
        path: Image path (s3://, http://, https://, or data:)
    
    Returns:
        bytes: Image data
        
    Raises:
        ValueError: If path format is invalid
        requests.HTTPError: If HTTP request fails
    """
    if path.startswith("data:"):
        # Base64 Data URI
        # Format: data:image/jpeg;base64,....
        header, encoded = path.split(",", 1)
        return base64.b64decode(encoded)
    
    elif path.startswith("http"):
        # It's a presigned URL or public URL
        resp = requests.get(path, timeout=10.0)
        resp.raise_for_status()
        return resp.content
    
    elif path.startswith("s3://"):
        # s3://bucket/key
        path_parts = path[5:].split("/", 1)
        if len(path_parts) != 2:
            raise ValueError(f"Invalid S3 URI: {path}")
        bucket, key = path_parts
        
        # Fetch from MinIO
        response = None
        try:
            response = minio_client.get_object(bucket, key)
            return response.read()
        finally:
            if response:
                response.close()
                response.release_conn()
    
    else:
        raise ValueError("Invalid image path format. Must be s3://, http(s)://, or data:")

def parse_image(image_bytes: bytes, ocr_url: str = OCR_URL, timeout: int = 60) -> str:
    """
    Sends image to NemoRetriever OCR v1 NIM and returns extracted text.

    The NIM accepts image URLs via INPUT_IMAGE_URLS. Since we have raw bytes,
    we encode the image as a base64 data URI and pass it as the URL.

    Response OUTPUT contains JSON with text regions.
    """
    if ocr_url is None:
        ocr_url = OCR_URL

    # Encode image as base64 data URI to use as the image URL
    b64 = base64.b64encode(image_bytes).decode('utf-8')
    image_url = f"data:image/jpeg;base64,{b64}"

    payload = {
        "inputs": [
            {
                "name": "INPUT_IMAGE_URLS",
                "shape": [1, 1],
                "datatype": "BYTES",
                "data": [[image_url]]
            },
            {
                "name": "MERGE_LEVELS",
                "shape": [1, 1],
                "datatype": "BYTES",
                "data": [["word"]]
            }
        ],
        "outputs": [{"name": "OUTPUT"}]
    }

    try:
        print(f"Sending request to {ocr_url}...")
        resp = requests.post(ocr_url, json=payload, timeout=timeout)
        resp.raise_for_status()
        data = resp.json()
        print(f"✅ OCR Response received")
        print(f"\n🔍 DEBUG - Full response structure:")
        print(json.dumps(data, indent=2)[:1000])

        # OUTPUT is a JSON string containing list of text regions
        # e.g. [{"text": "...", "confidence": 0.99, "bounding_box": [...]}]
        outputs = data.get('outputs', [])
        if not outputs:
            print("⚠️  No outputs in response")
            return ""

        raw = outputs[0].get('data', [])
        # raw is a nested list [[json_string, ...]] or flat list
        if isinstance(raw[0], list):
            raw = raw[0]

        all_text = []
        for item in raw:
            try:
                regions = json.loads(item) if isinstance(item, str) else item
                if isinstance(regions, list):
                    for r in regions:
                        t = r.get('text', '').strip()
                        if t:
                            all_text.append(t)
                elif isinstance(regions, dict) and regions.get('text'):
                    all_text.append(regions['text'].strip())
            except (json.JSONDecodeError, AttributeError):
                if str(item).strip():
                    all_text.append(str(item).strip())

        extracted = "\n".join(all_text)
        print(f"✅ Extracted {len(all_text)} text regions, {len(extracted)} chars")
        return extracted

    except requests.HTTPError as e:
        print(f"❌ HTTP Error: {e}")
        print(f"Response: {e.response.text if hasattr(e, 'response') else 'No response'}")
        raise
    except Exception as e:
        print(f"❌ OCR request failed: {e}")
        raise

### List Minio Storage Bucket

In [ ]:
# Initialize MinIO client
minio_client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

# Ensure bucket exists
if not minio_client.bucket_exists(MINIO_BUCKET):
    minio_client.make_bucket(MINIO_BUCKET)
    print(f"✅ Created bucket: {MINIO_BUCKET}")
else:
    print(f"✅ Bucket exists: {MINIO_BUCKET}")

# List existing objects
print("\n📁 Current objects in bucket:")
objects = list(minio_client.list_objects(MINIO_BUCKET, recursive=True))
for obj in objects:
    print(f"  - {obj.object_name}")
print(f"\nTotal objects: {len(objects)}")

### Upload id and photo to minio storage

In [7]:
test_id_image_path = "data/id16.jpg"
test_face_image_path = "data/rb.jpg"

In [ ]:
# Upload a test ID card image
object_name = os.path.basename(test_id_image_path)

if os.path.exists(test_id_image_path):
    # Read the image
    with open(test_id_image_path, "rb") as f:
        image_data = f.read()
    
    # Upload to MinIO
    minio_client.put_object(
        MINIO_BUCKET,
        object_name,
        io.BytesIO(image_data),
        len(image_data),
        content_type="image/jpeg"
    )
    
    print(f"✅ Uploaded: {object_name}")
    print(f"   Size: {len(image_data)} bytes")
    
    # Generate S3 URI
    id_s3_uri = f"s3://{MINIO_BUCKET}/{object_name}"
    print(f"   S3 URI: {id_s3_uri}")
    
    # Generate presigned URL for viewing
    presigned_url = minio_client.presigned_get_object(MINIO_BUCKET, object_name)
    print(f"   Presigned URL: {presigned_url}...")
else:
    print(f"❌ Test image not found: {test_id_image_path}")
    s3_uri = None

In [ ]:
# Upload a reference face photo
face_object_name = os.path.basename(test_face_image_path)

if os.path.exists(test_face_image_path):
    # Upload to MinIO
    with open(test_face_image_path, "rb") as f:
        face_data = f.read()
    
    minio_client.put_object(
        MINIO_BUCKET,
        face_object_name,
        io.BytesIO(face_data),
        len(face_data),
        content_type="image/jpeg"
    )
    
    print(f"✅ Uploaded face image: {face_object_name}")
    face_s3_uri = f"s3://{MINIO_BUCKET}/{face_object_name}"
    
    # Get face embedding with anti-spoofing
    print("\n🔍 Generating face embedding...")
    files = {'file': ('face.jpg', face_data, 'image/jpeg')}
    embed_resp = requests.post(FACE_EMBEDDING_URL, files=files, timeout=30)
    
    if embed_resp.status_code == 200:
        embed_data = embed_resp.json()
        print("✅ Face Embedding Generated:")
        print(f"   Embedding dimension: {len(embed_data.get('embedding', []))}")
        print(f"   Confidence: {embed_data.get('confidence', 0):.4f}")
        print(f"   Is Live: {embed_data.get('is_live', True)}")
        print(f"   Anti-spoof score: {embed_data.get('antispoof_score', 0):.4f}")
        
        face_embedding = embed_data.get('embedding')
    else:
        print(f"❌ Embedding failed: {embed_resp.text}")
        face_embedding = None
else:
    print(f"❌ Test face image not found: {test_face_image_path}")
    face_s3_uri = None
    face_embedding = None

In [10]:
id_image_data  = get_image_bytes(minio_client, id_s3_uri)
print(f"Retrieved {len(id_image_data)} bytes")

Retrieved 200729 bytes


In [ ]:
ocr_extracted_text = parse_image(id_image_data, ocr_url=OCR_URL, timeout=60)
print("Extracted text:", ocr_extracted_text)

In [ ]:
# Test health endpoint first and parse structured ocr output
health_resp = requests.get(f"{KYC_VERIFIER_URL}/health")
print("🏥 KYC Verifier Health:")
print(json.dumps(health_resp.json(), indent=2))

# Parse the ID document
if id_s3_uri:
    print("\n📄 Parsing ID document...")
    parse_payload = {
        "id_image_s3": id_s3_uri
    }
    
    parse_resp = requests.post(
        f"{KYC_VERIFIER_URL}/parse",
        json=parse_payload,
        timeout=60
    )
    
    print(f"Status: {parse_resp.status_code}")
    
    if parse_resp.status_code == 200:
        parsed_data = parse_resp.json()
        print("\n✅ Parsed ID Data:")
        print(json.dumps(parsed_data, indent=2))
    else:
        print(f"❌ Parse failed: {parse_resp.text}")
        parsed_data = None
else:
    print("⚠️ Skipping parse - no S3 URI available")
    parsed_data = None

### Ingesting data to db

1. ID to postgress

In [ ]:
# Connect to PostgreSQL
try:
    pg_conn = psycopg2.connect(POSTGRES_DSN)
    cursor = pg_conn.cursor()
    print("✅ Connected to PostgreSQL")
    
    # Check table structure
    cursor.execute("""
        SELECT column_name, data_type 
        FROM information_schema.columns 
        WHERE table_name = 'kyc_users'
        ORDER BY ordinal_position;
    """)
    
    print("\n📊 kyc_users table structure:")
    for col_name, col_type in cursor.fetchall():
        print(f"   {col_name}: {col_type}")
    
    # Count existing users
    cursor.execute("SELECT COUNT(*) FROM kyc_users;")
    count = cursor.fetchone()[0]
    print(f"\n📈 Current user count: {count}")
    
except Exception as e:
    print(f"❌ PostgreSQL error: {e}")
    pg_conn = None

In [14]:
assert type(parsed_data)==dict
assert type(parsed_data['parsed_id']) == dict

In [ ]:
# Insert test user data
if pg_conn and parsed_data:
    try:
        # Use parsed data or defaults
        user_data = {
            'name': parsed_data['parsed_id'].get('name', 'Test User'),
            'dl_no': parsed_data['parsed_id'].get('dl_no', 'TEST123456'),
            'dob': parsed_data['parsed_id'].get('dob', '1990-01-01'),
            'address': parsed_data['parsed_id'].get('address', 'Test Address')
        }
        
        print(f"\n💾 Inserting user: {user_data['name']}")
        
        cursor.execute("""
            INSERT INTO kyc_users (name, dl_no, dob, address)
            VALUES (%s, %s, %s, %s)
            ON CONFLICT (dl_no) DO UPDATE SET 
                name = EXCLUDED.name, 
                dob = EXCLUDED.dob, 
                address = EXCLUDED.address
            RETURNING user_id, created_at;
        """, (
            user_data['name'],
            user_data['dl_no'],
            user_data['dob'],
            user_data['address']
        ))
        
        result = cursor.fetchone()
        user_id = result[0]
        pg_conn.commit()
        
        print(f"✅ User saved with ID: {user_id}")
        print(f"   Created: {result[1]}")
        # print(f"   Updated: {result[2]}")
        
    except Exception as e:
        print(f"❌ Insert failed: {e}")
        pg_conn.rollback()
        user_id = None
else:
    print("⚠️ Skipping insert - no database connection or parsed data")
    user_id = None

2. Photo to milvus

In [ ]:
# Connect to Milvus
try:
    connections.connect(alias="default", host=MILVUS_HOST, port=MILVUS_PORT)
    print("✅ Connected to Milvus")
    
    # Check if collection exists
    if utility.has_collection(MILVUS_COLLECTION):
        collection = Collection(MILVUS_COLLECTION)
        collection.load()
        
        print(f"\n📊 Collection: {MILVUS_COLLECTION}")
        print(f"   Entities: {collection.num_entities}")
        print(f"   Schema: {collection.schema}")
    else:
        print(f"⚠️ Collection '{MILVUS_COLLECTION}' does not exist")
        print("   Run kyc-verifier service once to create it")
        collection = None
        
except Exception as e:
    print(f"❌ Milvus error: {e}")
    collection = None

In [ ]:
# Insert face embedding into Milvus
if collection and user_id and face_embedding:
    try:
        # Delete existing entry for this user (if any)
        try:
            collection.delete(f"user_id == {user_id}")
            print(f"🗑️ Deleted old embeddings for user_id {user_id}")
        except:
            pass
        
        # Insert new embedding
        data = [
            [user_id],              # user_id
            [face_object_name],     # image_id
            [face_embedding],       # embedding
            ["facenet-v1"]          # embedding_model_version  
        ]
        
        result = collection.insert(data)
        collection.flush()
        
        print(f"\n✅ Inserted embedding for user_id {user_id}")
        print(f"   Insert count: {len(result.primary_keys)}")
        print(f"   Total entities: {collection.num_entities}")
        
    except Exception as e:
        print(f"❌ Milvus insert failed: {e}")
else:
    print("⚠️ Skipping Milvus insert - missing collection, user_id, or embedding")

### . Full KYC Verification Flow

Test the complete `/verify` endpoint that orchestrates all steps

In [ ]:
# Run full verification
if id_s3_uri and face_s3_uri:
    print("\n🔐 Running full KYC verification...\n")
    
    verify_payload = {
        "client_request_id": f"test-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
        "live_image_s3": face_s3_uri,
        "id_image_s3": id_s3_uri,
        # Optional: skip re-parsing if we already have parsed data
        # "parsed_id_override": parsed_data
    }
    
    print("📤 Request payload:")
    print(json.dumps(verify_payload, indent=2))
    
    verify_resp = requests.post(
        f"{KYC_VERIFIER_URL}/verify",
        json=verify_payload,
        timeout=120
    )
    
    print(f"\n📥 Response Status: {verify_resp.status_code}\n")
    
    if verify_resp.status_code == 200:
        verify_result = verify_resp.json()
        print("✅ VERIFICATION RESULT:")
        print(json.dumps(verify_result, indent=2))
        
        # Highlight key results
        print("\n" + "="*50)
        print("KEY RESULTS:")
        print("="*50)
        print(f"Status: {verify_result.get('status')}")
        print(f"User ID: {verify_result.get('user_id')}")
        print(f"Match Score: {verify_result.get('match_score', 0):.4f}")
        print(f"Is Live: {verify_result.get('is_live')}")
        print(f"Message: {verify_result.get('message')}")
        
        if verify_result.get('parsed_id'):
            print("\nParsed ID Info:")
            for key, value in verify_result['parsed_id'].items():
                print(f"  {key}: {value}")
    else:
        print(f"❌ Verification failed:")
        print(verify_resp.text)
else:
    print("⚠️ Skipping verification - missing S3 URIs")

In [ ]:
results = collection.query(
    expr="pk >= 0",
    output_fields=["pk", "user_id", "image_id"]
)
print(results)

In [ ]:
# Close database connections
if pg_conn:
    cursor.close()
    pg_conn.close()
    print("✅ Closed PostgreSQL connection")

if collection:
    connections.disconnect("default")
    print("✅ Closed Milvus connection")

print("\n🎉 All done!")